In [1]:
!nvidia-smi

Sun Oct  4 11:20:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Wed_Aug_20_01:58:59_PM_PDT_2025
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [3]:
%%writefile tensor_core_matmul.cu

#include <stdio.h>
#include <stdlib.h>
#include <math.h>
#include <cuda.h>
#include <cuda_fp16.h>
#include <mma.h>

using namespace nvcuda;

#define N 64
#define TILE 16

__global__ void matrixMulTensorCore(const half *A,
                                    const half *B,
                                    float *C)
{
    int tileRow = blockIdx.y;
    int tileCol = blockIdx.x;

    wmma::fragment<wmma::matrix_a,
                   TILE, TILE, TILE,
                   half,
                   wmma::row_major> a_frag;

    wmma::fragment<wmma::matrix_b,
                   TILE, TILE, TILE,
                   half,
                   wmma::row_major> b_frag;

    wmma::fragment<wmma::accumulator,
                   TILE, TILE, TILE,
                   float> c_frag;

    wmma::fill_fragment(c_frag, 0.0f);

    for (int k = 0; k < N; k += TILE)
    {
        const half *A_tile =
            A + tileRow * TILE * N + k;

        const half *B_tile =
            B + k * N + tileCol * TILE;

        wmma::load_matrix_sync(
            a_frag,
            A_tile,
            N
        );

        wmma::load_matrix_sync(
            b_frag,
            B_tile,
            N
        );

        wmma::mma_sync(
            c_frag,
            a_frag,
            b_frag,
            c_frag
        );
    }

    float *C_tile =
        C + tileRow * TILE * N + tileCol * TILE;

    wmma::store_matrix_sync(
        C_tile,
        c_frag,
        N,
        wmma::mem_row_major
    );
}

int main()
{
    half *h_A;
    half *h_B;
    float *h_C;

    half *d_A;
    half *d_B;
    float *d_C;

    size_t sizeA = N * N * sizeof(half);
    size_t sizeB = N * N * sizeof(half);
    size_t sizeC = N * N * sizeof(float);

    h_A = (half *)malloc(sizeA);
    h_B = (half *)malloc(sizeB);
    h_C = (float *)malloc(sizeC);

    for (int i = 0; i < N * N; i++)
    {
        h_A[i] = __float2half(1.0f);
        h_B[i] = __float2half(1.0f);
    }

    cudaMalloc((void **)&d_A, sizeA);
    cudaMalloc((void **)&d_B, sizeB);
    cudaMalloc((void **)&d_C, sizeC);

    cudaMemcpy(d_A, h_A, sizeA,
               cudaMemcpyHostToDevice);

    cudaMemcpy(d_B, h_B, sizeB,
               cudaMemcpyHostToDevice);

    dim3 block(32);
    dim3 grid(N / TILE, N / TILE);

    matrixMulTensorCore<<<grid, block>>>(
        d_A, d_B, d_C
    );

    cudaDeviceSynchronize();

    cudaError_t err = cudaGetLastError();

    if (err != cudaSuccess)
    {
        printf("CUDA Error: %s\n",
               cudaGetErrorString(err));
        return 1;
    }

    cudaMemcpy(h_C, d_C, sizeC,
               cudaMemcpyDeviceToHost);

    printf("C[0][0]   = %f\n", h_C[0]);
    printf("C[1][1]   = %f\n", h_C[N + 1]);
    printf("C[16][16] = %f\n", h_C[16 * N + 16]);
    printf("C[32][32] = %f\n", h_C[32 * N + 32]);
    printf("C[63][63] = %f\n", h_C[63 * N + 63]);

    bool correct = true;

    for (int i = 0; i < N * N; i++)
    {
        if (fabs(h_C[i] - 64.0f) > 0.001f)
        {
            correct = false;
            break;
        }
    }

    if (correct)
        printf("Matrix multiplication is CORRECT.\n");
    else
        printf("Matrix multiplication is INCORRECT.\n");

    cudaFree(d_A);
    cudaFree(d_B);
    cudaFree(d_C);

    free(h_A);
    free(h_B);
    free(h_C);

    return 0;
}

Writing tensor_core_matmul.cu


In [4]:
!nvcc -arch=sm_75 tensor_core_matmul.cu -o tensor_core_matmul

In [5]:
!./tensor_core_matmul

C[0][0]   = 64.000000
C[1][1]   = 64.000000
C[16][16] = 64.000000
C[32][32] = 64.000000
C[63][63] = 64.000000
Matrix multiplication is CORRECT.
